In [1]:
from astropy.table import Table, vstack, unique
import numpy as np
from tqdm import tqdm

from astropy import units as u

In [2]:
# join shredded and missing target ids
shredded1 = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs.fits')
shredded2 = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_missing.fits')
shredded = vstack((shredded1, shredded2), join_type = 'outer')
shredded[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64
1084427169955849,556089,242.72318746350186,53.635782093163655,0.03063192091855999,0.03063192091855999,0,789.0584922824055,0.0017610627660455075,0.32999999905624167,44.0717658996582,224.0717662682332,0.0,1,0.030631930371296624,3.0,42.73534606638948,12955.934441406864,0.030485035010313136
1084434300272644,375898,243.65642926326169,54.27724177905015,0.07850942434688926,0.07850942434688926,0,310.9083716990426,0.0018719092656299759,0.32999999823339154,54.93254852294922,54.93254783937784,0.0,1,0.07850942838567453,3.0,-186.32208858017614,40998.983929035065,0.11197763201881784
1084427178344451,21266,243.62537013307835,53.712014537285526,0.05723634298805001,0.05723634298805001,0,849.3460895419121,0.0015670324751866896,0.32999999454262413,111.70044708251953,111.70044970679623,0.0,1,0.05723634831154621,3.0,190.61562929764193,29488.240413800784,0.07999449571015632
1084434300272643,375898,243.65118121249577,54.275090804072406,0.07974131508417623,0.07974131508417623,0,1123.4490712519037,0.001871909265610506,0.3299999982299592,54.93254852294922,234.93254783934097,0.0,1,0.07974131906965695,3.0,155.89283818049722,41212.106251148354,0.11197763201881784
1084427178344450,21266,243.62044993111272,53.713173370492754,0.05591772316967483,0.05591772316967483,0,2463.962895900011,0.0015670324751904594,0.329999994543418,111.70044708251953,291.700449706759,0.0,1,0.05591772860512272,3.0,-183.53309221409478,29259.976909073113,0.07999449571015632


In [3]:
# remove duplicate target ids from shredded
shredded = unique(shredded, keys = 'TARGETID')

In [4]:
# sga ids to remove
id_to_remove = np.load('/pscratch/sd/d/dbustos/VI/ids_to_remove/sga_ids.npy').astype(int)

# target ids to remove
targs_to_remove = np.load('/pscratch/sd/d/dbustos/VI/ids_to_remove/target_ids.npy').astype(int)

In [5]:
shredded['selection'] = 0

In [6]:
for i in tqdm(id_to_remove):
    shredded['selection'][shredded['SGA_ID']==i] = 1

100%|██████████| 3560/3560 [00:00<00:00, 7671.94it/s]


In [7]:
for i in tqdm(targs_to_remove):
    shredded['selection'][shredded['TARGETID']==i] = 1

100%|██████████| 7661/7661 [00:00<00:00, 9426.60it/s]


In [8]:
# save table
shredded.write('/pscratch/sd/d/dbustos/rot_curves/shredded.fits',format='fits',overwrite=True)

In [9]:
# remove all fibers marked as 1 and save again
shredded_new = shredded[shredded['selection']==0]
#shredded_new.write('/pscratch/sd/d/dbustos/rot_curves/shredded_VI',format='fits',overwrite=True)

In [10]:
shredded_new = shredded[shredded['selection']==0]

In [11]:
shredded_new[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,selection
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1,0.010016039025980373,3.0,-2.9046128078129714,4248.5806387633265,0.010025824882245725,0
50491463565312,832982,186.082682159506,31.514254503332,0.0039163048042792185,0.0039163048042792185,0,53.936871471232735,0.03058363548599275,0.7147463528655635,106.88983154296875,103.84630531401453,3.043526228954221,1,0.003916374956159986,12.0,-66.42085753186161,1181405.1015997794,3.940741291696953,0
67792481026048,992860,129.121654946794,18.365939088565412,1.0791507885943028,1.0791507885943028,4,3.1017794758081436,0.006014833578063982,0.8697308071747932,33.59540939331055,32.12214705731604,1.4732623359945052,1,1.0791507896862822,4.0,315286.00740845513,323546.1670739308,0.013388979423911302,0
67798239805447,1002244,130.177241314472,18.6136575514083,-0.00199569129234795,4.13114935733491e-48,1570,1.94266889222573e+84,0.0019167662436021253,0.29353700413683614,2.2941808700561523,182.21476030528004,179.92057943522389,1,2.330288829665476e-05,3.0,-21285.694638192545,22268.499017273396,0.07427971360590893,0
67803973419009,741629,130.11975028080658,18.658106026168184,1.4718273175495524,1.4718273175495524,4,3.8873844295740128,0.005362295525560936,0.7916197954378289,29.529781341552734,28.084222118064,1.4455592234887327,1,1.4718273186811814,3.0,404513.6293253005,441519.6120455192,0.05214934911256348,0


In [12]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')

SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [13]:
def deproj_dist(proj_dist, angle, ba):
    q0 = .2

    rho = np.radians(angle)

    cos2i = (ba**2 - q0**2)/(1 - q0**2)

    x = np.cos(angle)

    y = np.sin(angle)

    deprojected = dist * np.sqrt((x**2) + (y**2/cos2i))

    return(deprojected)

In [14]:
# get deprojected distance
shredded_new['deproj_dist'] = np.nan
shredded_new['deproj_r26'] = np.nan


for i in tqdm(np.unique(shredded_new['SGA_ID'])):
    
    sga_idx = SGA_dict[i]

    targ_id = shredded_new['SGA_ID'] == i
    
    #targs = shredded_new[shredded_new['SGA_ID'] == i]

    dist = shredded_new['DIST'][targ_id]

    phi = shredded_new['ANGLE_OFF_AXIS'][targ_id]

    axis_ratio = SGA['BA'][sga_idx]

    r26_dist = .5 * SGA['D26'][sga_idx]

    dep = deproj_dist(dist, phi, axis_ratio)

    dep_r26 = dep*u.deg.to('arcmin')/r26_dist

    # deprojected distance in units degrees
    shredded_new['deproj_dist'][targ_id] = dep

    # deprojected distance in r26
    shredded_new['deproj_r26'][targ_id] = dep_r26

  0%|          | 0/7323 [00:00<?, ?it/s]/tmp/ipykernel_1014617/1084790706.py:12: RuntimeWarning: invalid value encountered in sqrt
  deprojected = dist * np.sqrt((x**2) + (y**2/cos2i))
100%|██████████| 7323/7323 [00:02<00:00, 2677.29it/s]


In [15]:
shredded_new.rename_column('selection', 'VI')

In [16]:
# only take rows with deprojected r26 < 1.1 
shredded_new = shredded_new[shredded_new['deproj_r26'] < 1.1]
shredded_new[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1,0.010016039025980373,3.0,-2.9046128078129714,4248.5806387633265,0.010025824882245725,0,0.002881332554681444,0.4760043290496788
50491463565312,832982,186.082682159506,31.514254503332,0.0039163048042792185,0.0039163048042792185,0,53.936871471232735,0.03058363548599275,0.7147463528655635,106.88983154296875,103.84630531401453,3.043526228954221,1,0.003916374956159986,12.0,-66.42085753186161,1181405.1015997794,3.940741291696953,0,0.0330088824030436,0.7714249118797284
67815440646152,993764,131.54147036553024,19.348963933595506,0.6888593799981794,0.6888593799981794,4,7.544203393161297,0.004898285806031614,0.44172182227817786,9.1672945022583,188.95042393619124,179.78312943393294,1,0.688859381126886,3.0,199184.86718681874,206561.79926844206,0.014690024421240908,0,0.00677902926522777,0.6113251204383032
169678387281926,1430874,257.7495971,56.9324928,0.028921517883227336,0.028921517883227336,0,121857.89663615823,0.003250448915035935,0.31250071154567677,6.557606220245361,206.22810010342067,199.6704938831753,1,0.02892152786179073,3.0,158.1223507935872,12147.438163427121,0.028379120392506063,0,0.008096415535868652,0.7783957484224773
234526940856326,5001181,151.1772061023157,1.833743864948717,0.03108437229025544,0.03108437229025544,0,13100.219412088394,0.001016580293940661,0.34937980530094326,107.75557708740234,118.54239042958262,10.786813342180281,1,0.031084381613582526,4.0,20.42217921981953,20799.955384114714,0.06202829814864271,0,0.0031839814928551905,1.0942754258430336


In [17]:
#target distance that can be considered in center
#units R26
center_dist_lim = 0.001

#minimum distance between two targets to be considered unique
#units R26
unique_dist_lim = 0.01

In [18]:
# redo criteria cut
#empty column to classify galaxy
shredded_new['Selection'] = 0
# test_counter = 0

#for each unique SGA ID
for i in tqdm(np.unique(shredded_new['SGA_ID'])):

    #identify all the galaxies and targets
    obs_id = shredded_new['SGA_ID'] == i
    
    #makes a table of the targets corresponding to this galaxy
    obs = shredded_new[obs_id]

    sga_id = SGA_dict[i] 

    criteria_a = False
    criteria_b = False


#if the galaxy has more than three observations
    if len(obs) >= 3:

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(obs['deproj_r26'] <= center_dist_lim):
            # test_counter += 1
        
            # check how many unique points on semimajor axis
            not_center = obs[obs['deproj_r26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['deproj_r26']) - np.min(not_center['deproj_r26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                shredded_new['Selection'][obs_id] = 1
                criteria_a = True

# #----
# # b
# #----
        elif len(obs) >= 10:
            distances = sorted(obs['deproj_r26'])
            counter = 0

            for i in range(len(distances)-1):
                if distances[i+1] - distances[i] > unique_dist_lim:
                    counter += 1
            
            if counter >= 10:
                shredded_new['Selection'][obs_id] = 2
                criteria_b = True

        
#----
# c
#----
          
        elif not (criteria_a or criteria_b):
            
            # identify all points on semimajor axis
            on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

            # identify all points off major axis
            off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]


            if len(on_major) > 0:
            
                #split targets on semimajor axis onto either side of center galaxy
                if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                    left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

                else:
                    left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
                left = on_major[left_index]
                right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)
                            or np.any(np.abs(off_major['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)):

                            #viable galaxy
                            shredded_new['Selection'][obs_id] = 3
                            
    # if tf_loa['Selection'][obs_id][0] == 1:
    #     print(obs['SGA_ID'][0])

# print(len(shredded_new[shredded_new['Selection']!=0]))
# print(test_counter)

100%|██████████| 7320/7320 [00:07<00:00, 942.99it/s] 


In [19]:
test = shredded_new[shredded_new['Selection']!=0]
test[test['SGA_ID']==7880]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64
2323933411934213,7880,185.733125,15.8228611111,0.005323099033745547,0.005323099033745547,0,3728.360779821822,0.0043905928533583575,0.06106110305626584,79.14043426513672,88.27983059302592,9.139396327889202,2,0.005323150790775155,31.0,28.722799430822043,200828.68909687627,0.6698712497341975,0,0.004466548449012998,0.062117437043239944
2323933411934216,7880,185.728458333,15.8218055556,0.005417802701625102,0.005417802701625102,0,70600.11193340678,0.0009293356022407178,0.012924509030454105,79.14043426513672,186.2534745952666,107.11304033012988,2,0.005417853563527056,31.0,56.966620304892984,200828.91670774587,0.6698712497341975,0,0.0009469101580256737,0.013168922893864931
2394302156111875,7880,185.80196675959266,15.83626408512994,0.7220624670555146,8.719948552727319e-05,4,1.6276874840259552,0.07190490491941197,0.9999999900188543,79.14043426513672,79.14043177022262,0.0,2,9.602362230614004e-05,32.0,213784.52315084724,200822.35056461443,0.6698712497341975,0,0.07190490491941197,0.9999999900188543
2403098244939780,7880,185.69920501863194,15.81730856404957,0.005233411271659658,0.005233411271659658,0,1210.5332055058097,0.028761961967746108,0.399999996007282,79.14043426513672,259.1404317701243,0.0,2,0.0052334639062756785,31.0,1.9748904673015313,200828.47724149766,0.6698712497341975,0,0.028761961967746108,0.399999996007282
2403098249134085,7880,185.75792365333908,15.828146212260327,0.005335280298157383,0.005335280298157383,0,358.62099460565514,0.028761961967761394,0.3999999960074946,79.14043426513672,79.14043177018168,0.0,2,0.005335331938270281,31.0,32.35566242089881,200828.7181484199,0.6698712497341975,0,0.028761961967761394,0.3999999960074946
2407496291450885,7880,185.6698480625189,15.81188382174219,1.1987306604563663,1.1987306604563663,4,2.021702378988266,0.05752392393556054,0.7999999920155142,79.14043426513672,259.14043177010944,0.0,2,1.1987306615557445,31.0,355943.0199886659,411675.48909894447,0.6698712497341975,0,0.05752392393556054,0.7999999920155142
2407496295645188,7880,185.74324340450644,15.825438280268983,0.005227759731996915,0.005227759731996915,0,1802.7238841954313,0.01438098098387043,0.1999999980036045,79.14043426513672,79.14043177015824,0.0,2,0.005227812422921248,31.0,0.28941114568068976,200828.46401230758,0.6698712497341975,0,0.01438098098387043,0.1999999980036045
2407496295645190,7880,185.78728533094431,15.833559114894939,0.48696119186510456,0.48696119186510456,0,21.72160205245018,0.057523923935522414,0.7999999920149838,79.14043426513672,79.1404317702086,0.0,2,0.486961193102845,31.0,143669.410902069,248277.87936924736,0.6698712497341975,0,0.057523923935522414,0.7999999920149838
2407496295645191,7880,185.71388408709106,15.820019455959304,0.005135495122947853,0.005135495122947853,0,9416.193258767496,0.014380980983892198,0.19999999800390722,79.14043426513672,259.140431770162,0.0,2,0.005135548750664927,31.0,-27.227000302248907,200828.25005962054,0.6698712497341975,0,0.014380980983892198,0.19999999800390722


In [22]:
# save only good galaxies
test = shredded_new[shredded_new['Selection']!=0]
test.write('/pscratch/sd/d/dbustos/rot_curves/shredded_VI.fits',format='fits',overwrite=True)

In [21]:
# test = shredded_new[shredded_new['Selection']!=0]
# print(len(np.unique(test['SGA_ID'])))